# Phase 2.2: v2a-RSN Depth Selection

Apply three depth-selection rules from Phase 2.1 to v2a-RSN c-GC and c-GC-star outputs.

For each fish + method combination:
- Load D_p instability values from summary.json
- Apply absolute threshold rule (epsilon = 0.01)
- Apply relative threshold rule (fraction = 0.1)
- Apply bootstrap-band rule (if available)
- Store results in structured CSV and JSON outputs
- Generate visualization

**Output:**
- `outputs/v2a-RSNs/depth_selection/depth_selection_summary.csv` (n_fish × 2 methods rows)
- `outputs/v2a-RSNs/depth_selection/depth_selection.json`
- `outputs/v2a-RSNs/depth_selection/depth_selection_plot.png`

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set up project paths
CAUSALISED_GC_RELATIVE_PATH = Path('src/markovianity_diagnostic/core/causalised-GC.py')
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / CAUSALISED_GC_RELATIVE_PATH).exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find '{CAUSALISED_GC_RELATIVE_PATH}' from {Path.cwd().resolve()}"
    )

# Add to path for imports
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.depth_selection import DepthSelector

print(f'Project root: {PROJECT_ROOT}')

## Load c-GC and c-GC-star outputs

In [ ]:
# Paths to analysis outputs
c_gc_summary_path = PROJECT_ROOT / 'outputs' / 'v2a-RSNs' / 'c-GC' / 'summary.json'
c_gc_star_summary_path = PROJECT_ROOT / 'outputs' / 'v2a-RSNs' / 'c-GC-star' / 'summary.json'

# Load summary data
with open(c_gc_summary_path) as f:
    c_gc_data = json.load(f)

with open(c_gc_star_summary_path) as f:
    c_gc_star_data = json.load(f)

print(f"Loaded c-GC data: {len(c_gc_data)} recordings")
print(f"Loaded c-GC-star data: {len(c_gc_star_data)} recordings")

# Show structure
print(f"\nFirst recording (c-GC):")
print(f"  Keys: {c_gc_data[0].keys()}")

## Extract fish identifiers and map to friendly names

In [ ]:
# Extract unique fish identifiers
def extract_fish_id(dataset_name: str) -> str:
    """Extract fish number from dataset name (e.g., '220119_F2_run11' -> 'F2')."""
    parts = dataset_name.split('_')
    for part in parts:
        if part.startswith('F') and len(part) == 2:
            return part
    return dataset_name

# Create fish mapping
c_gc_datasets = [entry['dataset'] for entry in c_gc_data]
fish_ids = sorted(set(extract_fish_id(d) for d in c_gc_datasets))
fish_mapping = {fid: f'fish-{i+1}' for i, fid in enumerate(fish_ids)}

print(f"Fish mapping: {fish_mapping}")

# Map each entry
for entry in c_gc_data:
    fish_id = extract_fish_id(entry['dataset'])
    entry['fish'] = fish_mapping[fish_id]
    
for entry in c_gc_star_data:
    fish_id = extract_fish_id(entry['dataset'])
    entry['fish'] = fish_mapping[fish_id]

print(f"\nMapped fish for c-GC: {[e['fish'] for e in c_gc_data]}")
print(f"Mapped fish for c-GC-star: {[e['fish'] for e in c_gc_star_data]}")

## Apply depth selection rules

In [ ]:
selector = DepthSelector()

# Store results
results_list = []

# Process c-GC
for entry in c_gc_data:
    fish = entry['fish']
    dataset = entry['dataset']
    
    # Extract D_p dictionary
    D_p = {int(k): v for k, v in entry.get('D_p', {}).items()}
    
    # Apply depth selection rules
    # Using default parameters from Phase 2 specification:
    # - absolute: epsilon=0.01, k_stable=2
    # - relative: fraction=0.1
    # - bootstrap: not available for real data
    result = selector.apply_all_rules(
        D_p=D_p,
        D_boot_pointwise={},  # No bootstrap data for real v2a data
        epsilon=0.01,
        k_stable=2,
        fraction=0.1,
        confidence=0.95,
    )
    
    results_list.append({
        'fish': fish,
        'method': 'c-GC',
        'dataset': dataset,
        'p_values': result.p_values,
        'D_p': D_p,
        'selected_absolute': result.selected['absolute'],
        'selected_relative': result.selected['relative'],
        'selected_bootstrap': result.selected['bootstrap_band'],
        'warnings': result.warnings,
    })

# Process c-GC-star
for entry in c_gc_star_data:
    fish = entry['fish']
    dataset = entry['dataset']
    
    D_p = {int(k): v for k, v in entry.get('D_p', {}).items()}
    
    result = selector.apply_all_rules(
        D_p=D_p,
        D_boot_pointwise={},
        epsilon=0.01,
        k_stable=2,
        fraction=0.1,
        confidence=0.95,
    )
    
    results_list.append({
        'fish': fish,
        'method': 'c-GC-star',
        'dataset': dataset,
        'p_values': result.p_values,
        'D_p': D_p,
        'selected_absolute': result.selected['absolute'],
        'selected_relative': result.selected['relative'],
        'selected_bootstrap': result.selected['bootstrap_band'],
        'warnings': result.warnings,
    })

print(f"Applied depth selection to {len(results_list)} fish-method combinations")

## Create summary CSV

In [ ]:
# Create summary dataframe
summary_data = []
for result in results_list:
    summary_data.append({
        'fish': result['fish'],
        'method': result['method'],
        'dataset': result['dataset'],
        'p_selected_absolute': result['selected_absolute'],
        'p_selected_relative': result['selected_relative'],
        'p_selected_bootstrap': result['selected_bootstrap'],
        'max_depth': max(result['p_values']) if result['p_values'] else None,
        'max_D_p': max(result['D_p'].values()) if result['D_p'] else None,
        'n_warnings': len(result['warnings']),
    })

summary_df = pd.DataFrame(summary_data)

print("Depth Selection Summary:")
print(summary_df.to_string())

print(f"\nSummary shape: {summary_df.shape}")
print(f"Expected: {len(fish_ids)} fish × 2 methods = {len(fish_ids) * 2} rows")

## Save outputs

In [ ]:
# Create output directory
output_dir = PROJECT_ROOT / 'outputs' / 'v2a-RSNs' / 'depth_selection'
output_dir.mkdir(parents=True, exist_ok=True)

# Save summary CSV
csv_path = output_dir / 'depth_selection_summary.csv'
summary_df.to_csv(csv_path, index=False)
print(f"Saved summary CSV to: {csv_path}")

# Save detailed JSON
json_output = {
    'metadata': {
        'analysis': 'v2a-RSN depth selection (Phase 2.2)',
        'rules': ['absolute', 'relative', 'bootstrap_band'],
        'absolute_epsilon': 0.01,
        'absolute_k_stable': 2,
        'relative_fraction': 0.1,
        'n_fish': len(fish_ids),
        'n_methods': 2,
        'n_recordings': len(results_list),
    },
    'fish_mapping': fish_mapping,
    'results': [
        {
            'fish': r['fish'],
            'method': r['method'],
            'dataset': r['dataset'],
            'p_values': r['p_values'],
            'D_p': {str(k): v for k, v in r['D_p'].items()},
            'selected': {
                'absolute': r['selected_absolute'],
                'relative': r['selected_relative'],
                'bootstrap_band': r['selected_bootstrap'],
            },
            'warnings': r['warnings'],
        }
        for r in results_list
    ],
}

json_path = output_dir / 'depth_selection.json'
with open(json_path, 'w') as f:
    json.dump(json_output, f, indent=2)
print(f"Saved detailed JSON to: {json_path}")

## Create visualization

In [ ]:
# Create depth selection plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

fish_list = sorted(set(r['fish'] for r in results_list))
colors = {'c-GC': 'tab:blue', 'c-GC-star': 'tab:orange'}
markers = {'c-GC': 'o', 'c-GC-star': 's'}

for idx, fish in enumerate(fish_list):
    ax = axes[idx]
    
    # Get results for this fish from both methods
    fish_results = [r for r in results_list if r['fish'] == fish]
    
    for result in fish_results:
        method = result['method']
        p_vals = sorted(result['p_values'])
        d_vals = [result['D_p'][p] for p in p_vals]
        
        # Plot D_p trajectory
        ax.plot(
            p_vals, d_vals,
            marker=markers[method],
            color=colors[method],
            linestyle='-',
            linewidth=2,
            markersize=8,
            label=method,
        )
        
        # Mark selected depths with vertical lines
        abs_sel = result['selected_absolute']
        rel_sel = result['selected_relative']
        
        if abs_sel is not None:
            ax.axvline(abs_sel, color=colors[method], linestyle='--', alpha=0.5, linewidth=1)
    
    ax.set_xlabel('Conditioning depth p', fontsize=11)
    ax.set_ylabel('Instability D_p', fontsize=11)
    ax.set_title(f'{fish}', fontsize=12, fontweight='bold')
    ax.set_xticks(range(1, 8))
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
png_path = output_dir / 'depth_selection_plot.png'
plt.savefig(png_path, dpi=200, bbox_inches='tight')
print(f"Saved depth selection plot to: {png_path}")
plt.show()

## Summary and Verification

In [ ]:
print("\n" + "="*70)
print("PHASE 2.2 DEPTH SELECTION SUMMARY")
print("="*70)

print(f"\nOutput files:")
print(f"  CSV: {csv_path}")
print(f"  JSON: {json_path}")
print(f"  PNG: {png_path}")

print(f"\nData dimensions:")
print(f"  N fish: {len(fish_ids)}")
print(f"  N methods: 2 (c-GC, c-GC-star)")
print(f"  Total rows (CSV): {len(summary_df)}")

print(f"\nDepth selection results:")
print(f"  Absolute rule selected: {summary_df['p_selected_absolute'].notna().sum()} / {len(summary_df)}")
print(f"  Relative rule selected: {summary_df['p_selected_relative'].notna().sum()} / {len(summary_df)}")
print(f"  Bootstrap rule selected: {summary_df['p_selected_bootstrap'].notna().sum()} / {len(summary_df)}")

print(f"\nSelected depths (absolute rule):")
print(summary_df[['fish', 'method', 'p_selected_absolute', 'max_depth']].to_string(index=False))

print(f"\nFile verification:")
for fpath in [csv_path, json_path, png_path]:
    if fpath.exists():
        size_mb = fpath.stat().st_size / (1024 * 1024)
        print(f"  {fpath.name}: {size_mb:.2f} MB ✓")
    else:
        print(f"  {fpath.name}: MISSING ✗")

print(f"\n" + "="*70)
print("Phase 2.2 complete. CSV has correct shape:", summary_df.shape == (len(fish_ids) * 2, len(summary_df.columns)))